In [1]:
import sys
from pathlib import Path
import numpy as np

root = Path.cwd()
candidates = [root, root.parent, root.parent.parent]
for base in candidates:
    if (base / "efgp_eigenpro_py").exists():
        sys.path.insert(0, str(base))
        break
else:
    raise RuntimeError("cannot find efgp_eigenpro_py from current path")

from efgp_eigenpro_py.nufft_ops import (
    nufft1d1,
    nufft1d2,
    nufft2d1,
    nufft2d2,
    nufft3d1,
    nufft3d2,
)
from efgp_eigenpro_py.toeplitz import apply_toeplitz_fft_1d, apply_toeplitz_fft_nd
from efgp_eigenpro_py.kernels import make_squared_exponential, make_matern
from efgp_eigenpro_py.efgp_solver import EFGPSolver

np.set_printoptions(precision=3, suppress=True)


In [2]:
def direct_type1_1d(x, c, ms, isign):
    m = (ms - 1) // 2
    ks = np.arange(-m, m + 1)
    out = np.zeros(ms, dtype=np.complex128)
    for i, k in enumerate(ks):
        out[i] = np.sum(c * np.exp(1j * isign * k * x))
    return out


def direct_type2_1d(x, f, isign):
    ms = f.shape[0]
    m = (ms - 1) // 2
    ks = np.arange(-m, m + 1)
    out = np.zeros(x.shape[0], dtype=np.complex128)
    for j, xj in enumerate(x):
        out[j] = np.sum(f * np.exp(1j * isign * ks * xj))
    return out


def check_nufft1d1():
    rng = np.random.default_rng(0)
    n = 20
    m = 3
    ms = 2 * m + 1
    x = rng.uniform(-np.pi, np.pi, size=n)
    c = rng.normal(size=n) + 1j * rng.normal(size=n)
    ref = direct_type1_1d(x, c, ms, isign=-1)
    got = nufft1d1(x, c, ms, 1e-12, -1)
    err = np.linalg.norm(got - ref) / np.linalg.norm(ref)
    print("nufft1d1 relerr =", err)
    return err


def check_nufft1d2():
    rng = np.random.default_rng(1)
    n = 20
    m = 3
    ms = 2 * m + 1
    x = rng.uniform(-np.pi, np.pi, size=n)
    f = rng.normal(size=ms) + 1j * rng.normal(size=ms)
    ref = direct_type2_1d(x, f, isign=+1)
    got = nufft1d2(x, f, 1e-12, +1)
    err = np.linalg.norm(got - ref) / np.linalg.norm(ref)
    print("nufft1d2 relerr =", err)
    return err


def check_adjoint_1d():
    rng = np.random.default_rng(2)
    n = 20
    m = 4
    ms = 2 * m + 1
    x = rng.uniform(-np.pi, np.pi, size=n)
    c = rng.normal(size=n) + 1j * rng.normal(size=n)
    f = rng.normal(size=ms) + 1j * rng.normal(size=ms)
    lhs = np.vdot(nufft1d1(x, c, ms, 1e-12, -1), f)
    rhs = np.vdot(c, nufft1d2(x, f, 1e-12, +1))
    err = abs(lhs - rhs) / max(abs(lhs), abs(rhs), 1e-15)
    print("adjoint1d relerr =", err)
    return err


check_nufft1d1()
check_nufft1d2()
check_adjoint_1d()


nufft1d1 relerr = 2.119439984319284e-13
nufft1d2 relerr = 1.9502523736597973e-13
adjoint1d relerr = 1.9524709105570578e-16


np.float64(1.9524709105570578e-16)

In [3]:
def direct_type1_2d(x, y, c, ms, mt, isign):
    m1 = (ms - 1) // 2
    m2 = (mt - 1) // 2
    ks = np.arange(-m1, m1 + 1)
    ls = np.arange(-m2, m2 + 1)
    out = np.zeros((ms, mt), dtype=np.complex128)
    for i, k in enumerate(ks):
        for j, l in enumerate(ls):
            out[i, j] = np.sum(c * np.exp(1j * isign * (k * x + l * y)))
    return out


def direct_type2_2d(x, y, f, isign):
    ms, mt = f.shape
    m1 = (ms - 1) // 2
    m2 = (mt - 1) // 2
    ks = np.arange(-m1, m1 + 1)
    ls = np.arange(-m2, m2 + 1)
    out = np.zeros(x.shape[0], dtype=np.complex128)
    for idx, (xj, yj) in enumerate(zip(x, y)):
        phase = np.exp(1j * isign * (ks[:, None] * xj + ls[None, :] * yj))
        out[idx] = np.sum(f * phase)
    return out


def check_nufft2d1():
    rng = np.random.default_rng(3)
    n = 15
    m = 1
    ms = 2 * m + 1
    mt = ms
    x = rng.uniform(-np.pi, np.pi, size=n)
    y = rng.uniform(-np.pi, np.pi, size=n)
    c = rng.normal(size=n) + 1j * rng.normal(size=n)
    ref = direct_type1_2d(x, y, c, ms, mt, isign=-1)
    got = nufft2d1(x, y, c, ms, mt, 1e-12, -1)
    err = np.linalg.norm(got - ref) / np.linalg.norm(ref)
    print("nufft2d1 relerr =", err)
    return err


def check_nufft2d2():
    rng = np.random.default_rng(4)
    n = 15
    m = 1
    ms = 2 * m + 1
    mt = ms
    x = rng.uniform(-np.pi, np.pi, size=n)
    y = rng.uniform(-np.pi, np.pi, size=n)
    f = rng.normal(size=(ms, mt)) + 1j * rng.normal(size=(ms, mt))
    ref = direct_type2_2d(x, y, f, isign=+1)
    got = nufft2d2(x, y, f, 1e-12, +1)
    err = np.linalg.norm(got - ref) / np.linalg.norm(ref)
    print("nufft2d2 relerr =", err)
    return err


def check_adjoint_2d():
    rng = np.random.default_rng(5)
    n = 12
    m = 1
    ms = 2 * m + 1
    mt = ms
    x = rng.uniform(-np.pi, np.pi, size=n)
    y = rng.uniform(-np.pi, np.pi, size=n)
    c = rng.normal(size=n) + 1j * rng.normal(size=n)
    f = rng.normal(size=(ms, mt)) + 1j * rng.normal(size=(ms, mt))
    lhs = np.vdot(nufft2d1(x, y, c, ms, mt, 1e-12, -1), f)
    rhs = np.vdot(c, nufft2d2(x, y, f, 1e-12, +1))
    err = abs(lhs - rhs) / max(abs(lhs), abs(rhs), 1e-15)
    print("adjoint2d relerr =", err)
    return err


check_nufft2d1()
check_nufft2d2()
check_adjoint_2d()


nufft2d1 relerr = 1.6972076115387318e-13
nufft2d2 relerr = 3.098507820756463e-13
adjoint2d relerr = 8.426222411670404e-17


np.float64(8.426222411670404e-17)

In [4]:
def dense_toeplitz_1d(XtXcol, mtot):
    m = (mtot - 1) // 2
    ks = np.arange(-m, m + 1)
    T = np.zeros((mtot, mtot), dtype=np.complex128)
    for i, ki in enumerate(ks):
        for j, kj in enumerate(ks):
            diff = ki - kj
            idx = diff + 2 * m
            T[i, j] = XtXcol[idx]
    return T


def dense_toeplitz_2d(XtXcol, mtot):
    m = (mtot - 1) // 2
    idxs = [(i, j) for i in range(-m, m + 1) for j in range(-m, m + 1)]
    M = mtot * mtot
    T = np.zeros((M, M), dtype=np.complex128)
    for a, (i1, j1) in enumerate(idxs):
        for b, (i2, j2) in enumerate(idxs):
            di, dj = i1 - i2, j1 - j2
            T[a, b] = XtXcol[di + 2 * m, dj + 2 * m]
    return T


def check_toeplitz_1d():
    rng = np.random.default_rng(6)
    mtot = 5
    XtXcol = rng.normal(size=2 * mtot - 1) + 1j * rng.normal(size=2 * mtot - 1)
    Gf = np.fft.fftn(XtXcol)
    a = rng.normal(size=mtot) + 1j * rng.normal(size=mtot)
    ref = dense_toeplitz_1d(XtXcol, mtot) @ a
    got = apply_toeplitz_fft_1d(Gf, a, mtot)
    err = np.linalg.norm(got - ref) / np.linalg.norm(ref)
    print("toeplitz1d relerr =", err)
    return err


def check_toeplitz_2d():
    rng = np.random.default_rng(7)
    mtot = 3
    XtXcol = rng.normal(size=(2 * mtot - 1, 2 * mtot - 1)) + 1j * rng.normal(size=(2 * mtot - 1, 2 * mtot - 1))
    Gf = np.fft.fftn(XtXcol)
    a = rng.normal(size=mtot * mtot) + 1j * rng.normal(size=mtot * mtot)
    ref = dense_toeplitz_2d(XtXcol, mtot) @ a
    got = apply_toeplitz_fft_nd(Gf, a, mtot, dim=2)
    err = np.linalg.norm(got - ref) / np.linalg.norm(ref)
    print("toeplitz2d relerr =", err)
    return err


check_toeplitz_1d()
check_toeplitz_2d()


toeplitz1d relerr = 4.367188687217488e-16
toeplitz2d relerr = 3.5050362236545974e-16


np.float64(3.5050362236545974e-16)

In [5]:
def check_delta_toeplitz_1d():
    rng = np.random.default_rng(8)
    mtot = 5
    XtXcol = rng.normal(size=2 * mtot - 1) + 1j * rng.normal(size=2 * mtot - 1)
    Gf = np.fft.fftn(XtXcol)
    T = dense_toeplitz_1d(XtXcol, mtot)
    max_err = 0.0
    for j in range(mtot):
        e = np.zeros(mtot, dtype=np.complex128)
        e[j] = 1.0
        got = apply_toeplitz_fft_1d(Gf, e, mtot)
        ref = T[:, j]
        err = np.linalg.norm(got - ref) / (np.linalg.norm(ref) + 1e-15)
        max_err = max(max_err, err)
    print("toeplitz1d delta max relerr =", max_err)
    return max_err


def check_delta_toeplitz_2d():
    rng = np.random.default_rng(9)
    mtot = 3
    XtXcol = rng.normal(size=(2 * mtot - 1, 2 * mtot - 1)) + 1j * rng.normal(size=(2 * mtot - 1, 2 * mtot - 1))
    Gf = np.fft.fftn(XtXcol)
    T = dense_toeplitz_2d(XtXcol, mtot)
    M = mtot * mtot
    max_err = 0.0
    for j in range(M):
        e = np.zeros(M, dtype=np.complex128)
        e[j] = 1.0
        got = apply_toeplitz_fft_nd(Gf, e, mtot, dim=2)
        ref = T[:, j]
        err = np.linalg.norm(got - ref) / (np.linalg.norm(ref) + 1e-15)
        max_err = max(max_err, err)
    print("toeplitz2d delta max relerr =", max_err)
    return max_err


check_delta_toeplitz_1d()
check_delta_toeplitz_2d()


toeplitz1d delta max relerr = 3.427028508998742e-16
toeplitz2d delta max relerr = 3.297238218807449e-16


np.float64(3.297238218807449e-16)

In [6]:
def mode_indices(mtot, dim):
    m = (mtot - 1) // 2
    grid = [np.arange(-m, m + 1)] * dim
    mesh = np.meshgrid(*grid, indexing="ij")
    return np.stack([g.reshape(-1) for g in mesh], axis=1)


def dense_phi(x, state, dim):
    tphx = 2.0 * np.pi * state.grid.h * (x - state.x_center)
    idx = mode_indices(state.grid.mtot, dim)
    phase = np.exp(1j * (tphx @ idx.T))
    return phase * state.weights.reshape(1, -1)


def check_rhs_dense(dim=1):
    rng = np.random.default_rng(10)
    n = 12
    x = rng.uniform(-1.0, 1.0, size=(n, dim))
    y = rng.normal(size=n) + 1j * rng.normal(size=n)
    kernel = make_squared_exponential(lengthscale=0.8, dim=dim, variance=1.0)
    solver = EFGPSolver(kernel, reg_lambda=1e-3, eps=1e-4, nufft_tol=1e-12)
    state = solver.precompute(x, y)
    phi = dense_phi(x, state, dim)
    ref = phi.conj().T @ y
    got = state.rhs
    err = np.linalg.norm(got - ref) / np.linalg.norm(ref)
    print("rhs vs dense relerr =", err)
    return err


def check_apply_A_dense(dim=1):
    rng = np.random.default_rng(11)
    n = 10
    x = rng.uniform(-1.0, 1.0, size=(n, dim))
    y = rng.normal(size=n) + 1j * rng.normal(size=n)
    kernel = make_squared_exponential(lengthscale=0.8, dim=dim, variance=1.0)
    solver = EFGPSolver(kernel, reg_lambda=1e-3, eps=1e-4, nufft_tol=1e-12)
    state = solver.precompute(x, y)
    phi = dense_phi(x, state, dim)
    A = phi.conj().T @ phi + solver.reg_lambda * np.eye(phi.shape[1])
    v = rng.normal(size=A.shape[0]) + 1j * rng.normal(size=A.shape[0])
    ref = A @ v
    got = solver._apply_A(state, v)
    err = np.linalg.norm(got - ref) / np.linalg.norm(ref)
    print("apply_A vs dense relerr =", err)
    return err


def check_predict_dense(dim=1):
    rng = np.random.default_rng(12)
    n = 10
    x_train = rng.uniform(-1.0, 1.0, size=(n, dim))
    y_train = rng.normal(size=n)
    x_test = rng.uniform(-1.0, 1.0, size=(5, dim))
    kernel = make_squared_exponential(lengthscale=0.8, dim=dim, variance=1.0)
    solver = EFGPSolver(kernel, reg_lambda=1e-3, eps=1e-4, nufft_tol=1e-12)
    state = solver.precompute(x_train, y_train)
    beta = rng.normal(size=state.weights.size) + 1j * rng.normal(size=state.weights.size)
    pred = solver.predict(x_test, beta, state)
    phi_test = dense_phi(x_test, state, dim)
    ref = np.real(phi_test @ beta)
    err = np.linalg.norm(pred - ref) / np.linalg.norm(ref)
    print("predict vs dense relerr =", err)
    return err


check_rhs_dense(dim=1)
check_apply_A_dense(dim=1)
check_predict_dense(dim=1)

# Optional: dim=2 (may be slower)
# check_rhs_dense(dim=2)
# check_apply_A_dense(dim=2)
# check_predict_dense(dim=2)


rhs vs dense relerr = 1.346748167084616e-13
apply_A vs dense relerr = 4.154140588439652e-14
predict vs dense relerr = 6.17893437097398e-13


np.float64(6.17893437097398e-13)

In [7]:
def direct_type1_3d(x, y, z, c, ms, mt, mu, isign):
    m1 = (ms - 1) // 2
    m2 = (mt - 1) // 2
    m3 = (mu - 1) // 2
    ks = np.arange(-m1, m1 + 1)
    ls = np.arange(-m2, m2 + 1)
    us = np.arange(-m3, m3 + 1)
    out = np.zeros((ms, mt, mu), dtype=np.complex128)
    for i, k in enumerate(ks):
        for j, l in enumerate(ls):
            for t, u in enumerate(us):
                out[i, j, t] = np.sum(c * np.exp(1j * isign * (k * x + l * y + u * z)))
    return out


def direct_type2_3d(x, y, z, f, isign):
    ms, mt, mu = f.shape
    m1 = (ms - 1) // 2
    m2 = (mt - 1) // 2
    m3 = (mu - 1) // 2
    ks = np.arange(-m1, m1 + 1)
    ls = np.arange(-m2, m2 + 1)
    us = np.arange(-m3, m3 + 1)
    out = np.zeros(x.shape[0], dtype=np.complex128)
    for idx, (xj, yj, zj) in enumerate(zip(x, y, z)):
        phase = np.exp(1j * isign * (ks[:, None, None] * xj + ls[None, :, None] * yj + us[None, None, :] * zj))
        out[idx] = np.sum(f * phase)
    return out


def check_nufft3d1():
    rng = np.random.default_rng(13)
    n = 9
    m = 1
    ms = 2 * m + 1
    mt = ms
    mu = ms
    x = rng.uniform(-np.pi, np.pi, size=n)
    y = rng.uniform(-np.pi, np.pi, size=n)
    z = rng.uniform(-np.pi, np.pi, size=n)
    c = rng.normal(size=n) + 1j * rng.normal(size=n)
    ref = direct_type1_3d(x, y, z, c, ms, mt, mu, isign=-1)
    got = nufft3d1(x, y, z, c, ms, mt, mu, 1e-12, -1)
    err = np.linalg.norm(got - ref) / np.linalg.norm(ref)
    print("nufft3d1 relerr =", err)
    return err


def check_nufft3d2():
    rng = np.random.default_rng(14)
    n = 9
    m = 1
    ms = 2 * m + 1
    mt = ms
    mu = ms
    x = rng.uniform(-np.pi, np.pi, size=n)
    y = rng.uniform(-np.pi, np.pi, size=n)
    z = rng.uniform(-np.pi, np.pi, size=n)
    f = rng.normal(size=(ms, mt, mu)) + 1j * rng.normal(size=(ms, mt, mu))
    ref = direct_type2_3d(x, y, z, f, isign=+1)
    got = nufft3d2(x, y, z, f, 1e-12, +1)
    err = np.linalg.norm(got - ref) / np.linalg.norm(ref)
    print("nufft3d2 relerr =", err)
    return err


def check_adjoint_3d():
    rng = np.random.default_rng(15)
    n = 9
    m = 1
    ms = 2 * m + 1
    mt = ms
    mu = ms
    x = rng.uniform(-np.pi, np.pi, size=n)
    y = rng.uniform(-np.pi, np.pi, size=n)
    z = rng.uniform(-np.pi, np.pi, size=n)
    c = rng.normal(size=n) + 1j * rng.normal(size=n)
    f = rng.normal(size=(ms, mt, mu)) + 1j * rng.normal(size=(ms, mt, mu))
    lhs = np.vdot(nufft3d1(x, y, z, c, ms, mt, mu, 1e-12, -1), f)
    rhs = np.vdot(c, nufft3d2(x, y, z, f, 1e-12, +1))
    err = abs(lhs - rhs) / max(abs(lhs), abs(rhs), 1e-15)
    print("adjoint3d relerr =", err)
    return err


check_nufft3d1()
check_nufft3d2()
check_adjoint_3d()


nufft3d1 relerr = 2.981492472287803e-13
nufft3d2 relerr = 1.907225260551651e-13
adjoint3d relerr = 4.170414942446225e-16


np.float64(4.170414942446225e-16)

In [8]:
def check_boundary_1d():
    eps = 1e-12
    x = np.array([-np.pi + eps, -np.pi / 2, 0.0, np.pi / 2, np.pi - eps])
    m = 2
    ms = 2 * m + 1
    rng = np.random.default_rng(20)
    c = rng.normal(size=x.size) + 1j * rng.normal(size=x.size)
    ref1 = direct_type1_1d(x, c, ms, isign=-1)
    got1 = nufft1d1(x, c, ms, 1e-12, -1)
    err1 = np.linalg.norm(got1 - ref1) / np.linalg.norm(ref1)
    f = rng.normal(size=ms) + 1j * rng.normal(size=ms)
    ref2 = direct_type2_1d(x, f, isign=+1)
    got2 = nufft1d2(x, f, 1e-12, +1)
    err2 = np.linalg.norm(got2 - ref2) / np.linalg.norm(ref2)
    print("boundary1d type1 relerr =", err1)
    print("boundary1d type2 relerr =", err2)
    return err1, err2


def check_boundary_2d():
    eps = 1e-12
    x = np.array([-np.pi + eps, -np.pi + eps, 0.0, 0.0, np.pi - eps, np.pi - eps])
    y = np.array([-np.pi + eps, np.pi - eps, -np.pi + eps, np.pi - eps, -np.pi + eps, np.pi - eps])
    m = 1
    ms = 2 * m + 1
    mt = ms
    rng = np.random.default_rng(21)
    c = rng.normal(size=x.size) + 1j * rng.normal(size=x.size)
    ref = direct_type1_2d(x, y, c, ms, mt, isign=-1)
    got = nufft2d1(x, y, c, ms, mt, 1e-12, -1)
    err = np.linalg.norm(got - ref) / np.linalg.norm(ref)
    print("boundary2d type1 relerr =", err)
    return err


def check_boundary_3d():
    eps = 1e-12
    vals = np.array([-np.pi + eps, np.pi - eps])
    grid = np.array(np.meshgrid(vals, vals, vals, indexing="ij"))
    x = grid[0].reshape(-1)
    y = grid[1].reshape(-1)
    z = grid[2].reshape(-1)
    m = 1
    ms = 2 * m + 1
    mt = ms
    mu = ms
    rng = np.random.default_rng(22)
    c = rng.normal(size=x.size) + 1j * rng.normal(size=x.size)
    ref = direct_type1_3d(x, y, z, c, ms, mt, mu, isign=-1)
    got = nufft3d1(x, y, z, c, ms, mt, mu, 1e-12, -1)
    err = np.linalg.norm(got - ref) / np.linalg.norm(ref)
    print("boundary3d type1 relerr =", err)
    return err


check_boundary_1d()
check_boundary_2d()
check_boundary_3d()


boundary1d type1 relerr = 4.4980720503662634e-13
boundary1d type2 relerr = 2.964606757256098e-13
boundary2d type1 relerr = 1.7456607336486327e-13
boundary3d type1 relerr = 2.4134604729795204e-13


np.float64(2.4134604729795204e-13)

In [9]:
def run_multi_seed(label, fn, n_trials=20):
    errs = []
    for seed in range(n_trials):
        rng = np.random.default_rng(seed)
        errs.append(float(fn(rng)))
    errs = np.array(errs)
    print(label, "mean =", errs.mean(), "max =", errs.max())
    return errs


def nufft1d1_once(rng):
    n = 20
    m = 3
    ms = 2 * m + 1
    x = rng.uniform(-np.pi, np.pi, size=n)
    c = rng.normal(size=n) + 1j * rng.normal(size=n)
    ref = direct_type1_1d(x, c, ms, isign=-1)
    got = nufft1d1(x, c, ms, 1e-12, -1)
    return np.linalg.norm(got - ref) / np.linalg.norm(ref)


def nufft1d2_once(rng):
    n = 20
    m = 3
    ms = 2 * m + 1
    x = rng.uniform(-np.pi, np.pi, size=n)
    f = rng.normal(size=ms) + 1j * rng.normal(size=ms)
    ref = direct_type2_1d(x, f, isign=+1)
    got = nufft1d2(x, f, 1e-12, +1)
    return np.linalg.norm(got - ref) / np.linalg.norm(ref)


def adjoint1d_once(rng):
    n = 20
    m = 4
    ms = 2 * m + 1
    x = rng.uniform(-np.pi, np.pi, size=n)
    c = rng.normal(size=n) + 1j * rng.normal(size=n)
    f = rng.normal(size=ms) + 1j * rng.normal(size=ms)
    lhs = np.vdot(nufft1d1(x, c, ms, 1e-12, -1), f)
    rhs = np.vdot(c, nufft1d2(x, f, 1e-12, +1))
    return abs(lhs - rhs) / max(abs(lhs), abs(rhs), 1e-15)


def nufft2d1_once(rng):
    n = 15
    m = 1
    ms = 2 * m + 1
    mt = ms
    x = rng.uniform(-np.pi, np.pi, size=n)
    y = rng.uniform(-np.pi, np.pi, size=n)
    c = rng.normal(size=n) + 1j * rng.normal(size=n)
    ref = direct_type1_2d(x, y, c, ms, mt, isign=-1)
    got = nufft2d1(x, y, c, ms, mt, 1e-12, -1)
    return np.linalg.norm(got - ref) / np.linalg.norm(ref)


def nufft2d2_once(rng):
    n = 15
    m = 1
    ms = 2 * m + 1
    mt = ms
    x = rng.uniform(-np.pi, np.pi, size=n)
    y = rng.uniform(-np.pi, np.pi, size=n)
    f = rng.normal(size=(ms, mt)) + 1j * rng.normal(size=(ms, mt))
    ref = direct_type2_2d(x, y, f, isign=+1)
    got = nufft2d2(x, y, f, 1e-12, +1)
    return np.linalg.norm(got - ref) / np.linalg.norm(ref)


def adjoint2d_once(rng):
    n = 12
    m = 1
    ms = 2 * m + 1
    mt = ms
    x = rng.uniform(-np.pi, np.pi, size=n)
    y = rng.uniform(-np.pi, np.pi, size=n)
    c = rng.normal(size=n) + 1j * rng.normal(size=n)
    f = rng.normal(size=(ms, mt)) + 1j * rng.normal(size=(ms, mt))
    lhs = np.vdot(nufft2d1(x, y, c, ms, mt, 1e-12, -1), f)
    rhs = np.vdot(c, nufft2d2(x, y, f, 1e-12, +1))
    return abs(lhs - rhs) / max(abs(lhs), abs(rhs), 1e-15)


def toeplitz1d_once(rng):
    mtot = 5
    XtXcol = rng.normal(size=2 * mtot - 1) + 1j * rng.normal(size=2 * mtot - 1)
    Gf = np.fft.fftn(XtXcol)
    a = rng.normal(size=mtot) + 1j * rng.normal(size=mtot)
    ref = dense_toeplitz_1d(XtXcol, mtot) @ a
    got = apply_toeplitz_fft_1d(Gf, a, mtot)
    return np.linalg.norm(got - ref) / np.linalg.norm(ref)


def toeplitz2d_once(rng):
    mtot = 3
    XtXcol = rng.normal(size=(2 * mtot - 1, 2 * mtot - 1)) + 1j * rng.normal(size=(2 * mtot - 1, 2 * mtot - 1))
    Gf = np.fft.fftn(XtXcol)
    a = rng.normal(size=mtot * mtot) + 1j * rng.normal(size=mtot * mtot)
    ref = dense_toeplitz_2d(XtXcol, mtot) @ a
    got = apply_toeplitz_fft_nd(Gf, a, mtot, dim=2)
    return np.linalg.norm(got - ref) / np.linalg.norm(ref)


def rhs_dense_once(rng):
    n = 8
    dim = 1
    x = rng.uniform(-1.0, 1.0, size=(n, dim))
    y = rng.normal(size=n) + 1j * rng.normal(size=n)
    kernel = make_squared_exponential(lengthscale=0.8, dim=dim, variance=1.0)
    solver = EFGPSolver(kernel, reg_lambda=1e-3, eps=1e-4, nufft_tol=1e-12)
    state = solver.precompute(x, y)
    phi = dense_phi(x, state, dim)
    ref = phi.conj().T @ y
    got = state.rhs
    return np.linalg.norm(got - ref) / np.linalg.norm(ref)


def apply_A_dense_once(rng):
    n = 8
    dim = 1
    x = rng.uniform(-1.0, 1.0, size=(n, dim))
    y = rng.normal(size=n) + 1j * rng.normal(size=n)
    kernel = make_squared_exponential(lengthscale=0.8, dim=dim, variance=1.0)
    solver = EFGPSolver(kernel, reg_lambda=1e-3, eps=1e-4, nufft_tol=1e-12)
    state = solver.precompute(x, y)
    phi = dense_phi(x, state, dim)
    A = phi.conj().T @ phi + solver.reg_lambda * np.eye(phi.shape[1])
    v = rng.normal(size=A.shape[0]) + 1j * rng.normal(size=A.shape[0])
    ref = A @ v
    got = solver._apply_A(state, v)
    return np.linalg.norm(got - ref) / np.linalg.norm(ref)


def predict_dense_once(rng):
    n = 8
    dim = 1
    x_train = rng.uniform(-1.0, 1.0, size=(n, dim))
    y_train = rng.normal(size=n)
    x_test = rng.uniform(-1.0, 1.0, size=(4, dim))
    kernel = make_squared_exponential(lengthscale=0.8, dim=dim, variance=1.0)
    solver = EFGPSolver(kernel, reg_lambda=1e-3, eps=1e-4, nufft_tol=1e-12)
    state = solver.precompute(x_train, y_train)
    beta = rng.normal(size=state.weights.size) + 1j * rng.normal(size=state.weights.size)
    pred = solver.predict(x_test, beta, state)
    phi_test = dense_phi(x_test, state, dim)
    ref = np.real(phi_test @ beta)
    return np.linalg.norm(pred - ref) / np.linalg.norm(ref)


run_multi_seed("nufft1d1", nufft1d1_once, n_trials=20)
run_multi_seed("nufft1d2", nufft1d2_once, n_trials=20)
run_multi_seed("adjoint1d", adjoint1d_once, n_trials=20)
run_multi_seed("nufft2d1", nufft2d1_once, n_trials=20)
run_multi_seed("nufft2d2", nufft2d2_once, n_trials=20)
run_multi_seed("adjoint2d", adjoint2d_once, n_trials=20)
run_multi_seed("toeplitz1d", toeplitz1d_once, n_trials=20)
run_multi_seed("toeplitz2d", toeplitz2d_once, n_trials=20)
run_multi_seed("rhs dense 1d se", rhs_dense_once, n_trials=10)
run_multi_seed("apply_A dense 1d se", apply_A_dense_once, n_trials=10)
run_multi_seed("predict dense 1d se", predict_dense_once, n_trials=10)


nufft1d1 mean = 2.0166193805028622e-13 max = 3.011638853323488e-13
nufft1d2 mean = 2.1594902560533346e-13 max = 3.0939760676656824e-13
adjoint1d mean = 3.804009968691558e-16 max = 1.3975204847205543e-15
nufft2d1 mean = 2.4925245740804895e-13 max = 3.84151792532926e-13
nufft2d2 mean = 2.25003160031009e-13 max = 3.6925354419245776e-13
adjoint2d mean = 3.469633969239783e-16 max = 7.788178292619029e-16
toeplitz1d mean = 3.7083765790033375e-16 max = 5.151481080000065e-16
toeplitz2d mean = 2.9575795691772833e-16 max = 4.3443225979877413e-16
rhs dense 1d se mean = 1.851088471653622e-13 max = 3.2354419899394123e-13
apply_A dense 1d se mean = 1.2883745782082856e-13 max = 2.412414132511281e-13
predict dense 1d se mean = 2.064351213965158e-13 max = 6.27342039022639e-13


array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [10]:
def check_rhs_dense_kernel(kernel, dim=1):
    rng = np.random.default_rng(30)
    n = 12
    x = rng.uniform(-1.0, 1.0, size=(n, dim))
    y = rng.normal(size=n) + 1j * rng.normal(size=n)
    solver = EFGPSolver(kernel, reg_lambda=1e-3, eps=1e-4, nufft_tol=1e-12)
    state = solver.precompute(x, y)
    phi = dense_phi(x, state, dim)
    ref = phi.conj().T @ y
    got = state.rhs
    err = np.linalg.norm(got - ref) / np.linalg.norm(ref)
    print("rhs dense kernel relerr =", err)
    return err


def check_apply_A_dense_kernel(kernel, dim=1):
    rng = np.random.default_rng(31)
    n = 10
    x = rng.uniform(-1.0, 1.0, size=(n, dim))
    y = rng.normal(size=n) + 1j * rng.normal(size=n)
    solver = EFGPSolver(kernel, reg_lambda=1e-3, eps=1e-4, nufft_tol=1e-12)
    state = solver.precompute(x, y)
    phi = dense_phi(x, state, dim)
    A = phi.conj().T @ phi + solver.reg_lambda * np.eye(phi.shape[1])
    v = rng.normal(size=A.shape[0]) + 1j * rng.normal(size=A.shape[0])
    ref = A @ v
    got = solver._apply_A(state, v)
    err = np.linalg.norm(got - ref) / np.linalg.norm(ref)
    print("apply_A dense kernel relerr =", err)
    return err


def check_predict_dense_kernel(kernel, dim=1):
    rng = np.random.default_rng(32)
    n = 10
    x_train = rng.uniform(-1.0, 1.0, size=(n, dim))
    y_train = rng.normal(size=n)
    x_test = rng.uniform(-1.0, 1.0, size=(5, dim))
    solver = EFGPSolver(kernel, reg_lambda=1e-3, eps=1e-4, nufft_tol=1e-12)
    state = solver.precompute(x_train, y_train)
    beta = rng.normal(size=state.weights.size) + 1j * rng.normal(size=state.weights.size)
    pred = solver.predict(x_test, beta, state)
    phi_test = dense_phi(x_test, state, dim)
    ref = np.real(phi_test @ beta)
    err = np.linalg.norm(pred - ref) / np.linalg.norm(ref)
    print("predict dense kernel relerr =", err)
    return err


matern_kernel = make_matern(lengthscale=0.8, nu=1.5, dim=1, variance=1.0)
check_rhs_dense_kernel(matern_kernel, dim=1)
check_apply_A_dense_kernel(matern_kernel, dim=1)
check_predict_dense_kernel(matern_kernel, dim=1)

# Optional: dim=2
# matern_kernel_2d = make_matern(lengthscale=0.8, nu=1.5, dim=2, variance=1.0)
# check_rhs_dense_kernel(matern_kernel_2d, dim=2)
# check_apply_A_dense_kernel(matern_kernel_2d, dim=2)
# check_predict_dense_kernel(matern_kernel_2d, dim=2)


rhs dense kernel relerr = 7.500435847359358e-14
apply_A dense kernel relerr = 5.2500811589315525e-14
predict dense kernel relerr = 1.2309392340409738e-13


np.float64(1.2309392340409738e-13)